# Metric signal analysis

This notebook asks whether implemented label-free metrics contain signal about labeled detector quality. It uses shared candidate pools, multiple datasets, and multiple random splits. Labels are used only for the final diagnostic comparisons.

## Design

The analysis avoids metric-specific search paths. Every candidate configuration receives every implemented label-free metric. Signal is then measured by rank correlation, top-candidate enrichment, selection regret, pairwise agreement, and within-model diagnostics.

In [1]:
import math
import os
import warnings
from html import escape

import numpy as np
import pandas as pd
from oddball import load
from pyod.models.iforest import IForest as PyodIForest
from pyod.models.knn import KNN
from pyod.models.lof import LOF as PyodLOF
from scipy.stats import kendalltau, spearmanr
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

from labelfree.metrics import (
    asoi_score,
    bounding_box_volume,
    excess_mass_auc,
    expected_anomaly_gap_score,
    ireos_score,
    laplacian_score,
    mass_volume_auc,
    normalized_pseudo_discrepancy_score,
    relative_top_median_score,
    score_cluster_metrics,
    sireos_score,
)
from labelfree.utils.validation import orient_scores

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

BASE_SEED = 7
SEEDS = list(range(1, 11))
DATASETS = ["cardio", "breastw", "satimage2", "annthyroid"]
DETECTOR_CONTAMINATION = 0.10
MAX_NORMAL = 500
MAX_ANOMALY = 120
REFERENCE_SIZE = 300
TOP_FRACTION = 0.20
BOOTSTRAP_ROUNDS = 600


class HtmlBlock(str):
    def _repr_html_(self):
        return str(self)

def html_table(rows, headers):
    head = "".join(f"<th style='text-align:left; padding:6px 8px; border-bottom:1px solid #d8dee4;'>{escape(str(header))}</th>" for header in headers)
    body = []
    for row in rows:
        body.append(
            "<tr style='border-bottom:1px solid #eaeef2;'>"
            + "".join(f"<td style='padding:6px 8px;'>{cell}</td>" for cell in row)
            + "</tr>"
        )
    return HtmlBlock(
        "<table style='border-collapse:collapse; width:100%; font-size:13px; color:#000; background:#fff;'>"
        f"<thead><tr>{head}</tr></thead><tbody>{''.join(body)}</tbody></table>"
    )


HtmlBlock(
    "<style>"
    ".output_html table,.output_html table.dataframe{background:#fff;color:#000;}"
    ".output_html table th,.output_html table td{background:#fff;color:#000;}"
    "</style>"
)


'<style>.output_html table,.output_html table.dataframe{background:#fff;color:#000;}.output_html table th,.output_html table td{background:#fff;color:#000;}</style>'

## Metric and selector registry

Lower-is-better metrics are negated for signal calculations so all signal values have the same polarity.

In [2]:
metric_info = pd.DataFrame(
    [
        ("rtm", "higher", "Relative top-median score separation"),
        ("eag", "higher", "Expected anomaly gap"),
        ("score_silhouette", "higher", "Silhouette of the score-induced split"),
        ("asoi", "higher", "Feature-space anomaly separation and overlap"),
        ("ireos", "higher", "RBF one-vs-rest separability proxy"),
        ("excess_mass", "higher", "Excess-mass criterion"),
        ("npd", "higher", "Generated-reference score discrepancy"),
        ("laplacian", "lower", "Graph smoothness of anomaly scores"),
        ("sireos", "lower", "Similarity-weighted isolation score"),
        ("mass_volume", "lower", "Mass-volume criterion"),
    ],
    columns=["method", "better", "description"],
)
metric_names = metric_info["method"].tolist()
metric_direction = dict(metric_info[["method", "better"]].to_records(index=False))
metric_info


,method,better,description
0,rtm,higher,Relative top-median score separation
1,eag,higher,Expected anomaly gap
2,score_silhouette,higher,Silhouette of the score-induced split
3,asoi,higher,Feature-space anomaly separation and overlap
4,ireos,higher,RBF one-vs-rest separability proxy
5,excess_mass,higher,Excess-mass criterion
6,npd,higher,Generated-reference score discrepancy
7,laplacian,lower,Graph smoothness of anomaly scores
8,sireos,lower,Similarity-weighted isolation score
9,mass_volume,lower,Mass-volume criterion


## Candidate pool

Every split uses the same family of detector configurations. This makes the question candidate-level: does a method rank already-observed configurations in a way that agrees with labeled holdout quality?

In [3]:
model_meta = {
    "sklearn_iforest": ("sklearn", "higher_is_normal"),
    "sklearn_ocsvm": ("sklearn", "higher_is_normal"),
    "sklearn_lof": ("sklearn", "higher_is_normal"),
    "pyod_iforest": ("pyod", "higher_is_anomalous"),
    "pyod_knn": ("pyod", "higher_is_anomalous"),
    "pyod_lof": ("pyod", "higher_is_anomalous"),
}


def sample_rows(values, max_count, seed):
    rng = np.random.default_rng(seed)
    count = min(max_count, len(values))
    return values[rng.choice(len(values), count, replace=False)]


def candidate_specs(n_train):
    max64 = min(64, n_train)
    max128 = min(128, n_train)
    return [
        ("sklearn_iforest", {"n_estimators": 80, "max_samples": max64}),
        ("sklearn_iforest", {"n_estimators": 160, "max_samples": max128}),
        ("sklearn_iforest", {"n_estimators": 240, "max_samples": max128}),
        ("sklearn_ocsvm", {"nu": 0.04, "gamma": 0.01}),
        ("sklearn_ocsvm", {"nu": 0.08, "gamma": 0.03}),
        ("sklearn_ocsvm", {"nu": 0.12, "gamma": 0.10}),
        ("sklearn_lof", {"n_neighbors": 10}),
        ("sklearn_lof", {"n_neighbors": 25}),
        ("sklearn_lof", {"n_neighbors": 40}),
        ("pyod_iforest", {"n_estimators": 80, "max_samples": max64}),
        ("pyod_iforest", {"n_estimators": 160, "max_samples": max128}),
        ("pyod_iforest", {"n_estimators": 240, "max_samples": max128}),
        ("pyod_knn", {"n_neighbors": 10}),
        ("pyod_knn", {"n_neighbors": 25}),
        ("pyod_knn", {"n_neighbors": 40}),
        ("pyod_lof", {"n_neighbors": 10}),
        ("pyod_lof", {"n_neighbors": 25}),
        ("pyod_lof", {"n_neighbors": 40}),
    ]


def build_model(model_name, params, seed):
    if model_name == "sklearn_iforest":
        return IsolationForest(contamination=DETECTOR_CONTAMINATION, random_state=seed, n_jobs=-1, **params)
    if model_name == "sklearn_ocsvm":
        return OneClassSVM(**params)
    if model_name == "sklearn_lof":
        return LocalOutlierFactor(contamination=DETECTOR_CONTAMINATION, novelty=True, **params)
    if model_name == "pyod_iforest":
        return PyodIForest(contamination=DETECTOR_CONTAMINATION, random_state=seed, **params)
    if model_name == "pyod_knn":
        return KNN(contamination=DETECTOR_CONTAMINATION, **params)
    return PyodLOF(contamination=DETECTOR_CONTAMINATION, **params)


detector_frame = pd.DataFrame(
    [{"model": name, "library": values[0], "score_polarity": values[1]} for name, values in model_meta.items()]
)
detector_frame


,model,library,score_polarity
0,sklearn_iforest,sklearn,higher_is_normal
1,sklearn_ocsvm,sklearn,higher_is_normal
2,sklearn_lof,sklearn,higher_is_normal
3,pyod_iforest,pyod,higher_is_anomalous
4,pyod_knn,pyod,higher_is_anomalous
5,pyod_lof,pyod,higher_is_anomalous


## Metric computation

Reference samples are drawn uniformly from the scaled normal-tuning bounding box. They are used only by metrics that explicitly need a background reference distribution.

In [4]:
def label_free_metric_values(X_scaled, scores, reference_scores, polarity):
    support_volume = bounding_box_volume(X_scaled)
    clusters = score_cluster_metrics(scores, contamination=DETECTOR_CONTAMINATION, score_polarity=polarity)
    return {
        "rtm": relative_top_median_score(scores, score_polarity=polarity),
        "eag": expected_anomaly_gap_score(scores, score_polarity=polarity),
        "score_silhouette": clusters["silhouette"],
        "asoi": asoi_score(X_scaled, scores, contamination=DETECTOR_CONTAMINATION, score_polarity=polarity),
        "ireos": ireos_score(
            X_scaled,
            scores,
            contamination=DETECTOR_CONTAMINATION,
            score_polarity=polarity,
            gamma_count=4,
        ),
        "excess_mass": excess_mass_auc(
            scores,
            reference_scores,
            support_volume=support_volume,
            level_count=80,
            score_polarity=polarity,
        ),
        "npd": normalized_pseudo_discrepancy_score(scores, reference_scores, score_polarity=polarity),
        "laplacian": laplacian_score(X_scaled, scores, n_neighbors=10, score_polarity=polarity),
        "sireos": sireos_score(X_scaled, scores, score_polarity=polarity),
        "mass_volume": mass_volume_auc(
            scores,
            reference_scores,
            support_volume=support_volume,
            alpha_min=0.80,
            alpha_max=0.98,
            alpha_count=80,
            score_polarity=polarity,
        ),
    }


def signal_value(metric, value):
    return value if metric_direction[metric] == "higher" else -value



## Evaluate candidates

The resulting table is the audit trail: one row per dataset, split seed, and candidate configuration. Label-free columns are computed without labels; labeled columns are held out for evaluation only.

In [5]:
rows = []
error_rows = []
dataset_rows = []

for dataset_index, dataset in enumerate(DATASETS):
    X, y = load(dataset)
    y = (y != 0).astype(int)
    for split_index, split_seed in enumerate(SEEDS):
        seed_base = split_seed + 100 * dataset_index
        normal = sample_rows(X[y == 0], MAX_NORMAL, seed_base)
        anomaly = sample_rows(X[y == 1], MAX_ANOMALY, seed_base + 10_000)

        X_train, X_rest = train_test_split(normal, test_size=0.40, random_state=split_seed)
        X_tune, X_hold_normal = train_test_split(X_rest, test_size=0.50, random_state=split_seed)
        X_hold = np.vstack([X_hold_normal, anomaly])
        y_hold = np.r_[np.zeros(len(X_hold_normal), dtype=int), np.ones(len(anomaly), dtype=int)]
        k = int(y_hold.sum())
        specs = candidate_specs(len(X_train))

        dataset_rows.append(
            {
                "dataset": dataset,
                "split_seed": split_seed,
                "features": X.shape[1],
                "train_normals": len(X_train),
                "tune_normals": len(X_tune),
                "holdout_normals": len(X_hold_normal),
                "holdout_anomalies": len(anomaly),
                "candidates": len(specs),
            }
        )

        for candidate_id, (model_name, params) in enumerate(specs):
            library, polarity = model_meta[model_name]
            try:
                pipe = make_pipeline(StandardScaler(), build_model(model_name, params, seed_base + candidate_id))
                pipe.fit(X_train)

                tune_scores = pipe.decision_function(X_tune)
                raw_holdout_scores = pipe.decision_function(X_hold)
                anomaly_scores = orient_scores(raw_holdout_scores, score_polarity=polarity)
                y_pred = np.zeros_like(y_hold)
                y_pred[np.argsort(anomaly_scores)[-k:]] = 1

                scaler = pipe.named_steps["standardscaler"]
                detector = pipe.steps[-1][1]
                X_tune_scaled = scaler.transform(X_tune)
                low = X_tune_scaled.min(axis=0)
                high = X_tune_scaled.max(axis=0)
                reference = np.random.default_rng(seed_base + 1000 + candidate_id).uniform(
                    low, high, size=(REFERENCE_SIZE, X_tune_scaled.shape[1])
                )
                reference_scores = detector.decision_function(reference)
                metrics = label_free_metric_values(X_tune_scaled, tune_scores, reference_scores, polarity)

                row = {
                    "dataset": dataset,
                    "split_seed": split_seed,
                    "candidate_id": candidate_id,
                    "model": model_name,
                    "library": library,
                    "params": params,
                    "score_polarity": polarity,
                    "roc_auc": roc_auc_score(y_hold, anomaly_scores),
                    "pr_auc": average_precision_score(y_hold, anomaly_scores),
                    "f1_at_k": f1_score(y_hold, y_pred),
                }
                row.update({f"raw_{name}": metrics[name] for name in metric_names})
                row.update({f"signal_{name}": signal_value(name, metrics[name]) for name in metric_names})
                rows.append(row)
            except Exception as exc:
                error_rows.append(
                    {
                        "dataset": dataset,
                        "split_seed": split_seed,
                        "candidate_id": candidate_id,
                        "model": model_name,
                        "error": repr(exc),
                    }
                )

candidate_results = pd.DataFrame(rows)
dataset_overview = pd.DataFrame(dataset_rows)
errors = pd.DataFrame(error_rows)

dataset_summary = dataset_overview.groupby("dataset", as_index=False).agg(
    splits=("split_seed", "nunique"),
    features=("features", "first"),
    train_normals=("train_normals", "mean"),
    tune_normals=("tune_normals", "mean"),
    holdout_normals=("holdout_normals", "mean"),
    holdout_anomalies=("holdout_anomalies", "mean"),
    candidate_evaluations=("candidates", "sum"),
)
dataset_summary = dataset_summary.astype(
    {
        "splits": int,
        "features": int,
        "train_normals": int,
        "tune_normals": int,
        "holdout_normals": int,
        "holdout_anomalies": int,
        "candidate_evaluations": int,
    }
)
dataset_summary


,dataset,splits,features,train_normals,tune_normals,holdout_normals,holdout_anomalies,candidate_evaluations
0,annthyroid,10,6,300,100,100,120,180
1,breastw,10,9,266,89,89,120,180
2,cardio,10,21,300,100,100,120,180
3,satimage2,10,36,300,100,100,71,180


## Candidate quality ranges

The supervised labels are used here only to show how much room each dataset leaves for candidate selection.

In [6]:
def candidate_quality_svg(results):
    by_dataset = results.groupby("dataset").agg(
        min_pr=("pr_auc", "min"),
        mean_pr=("pr_auc", "mean"),
        max_pr=("pr_auc", "max"),
        n=("pr_auc", "size"),
    ).loc[DATASETS]
    width, height = 780, 70 + 52 * len(by_dataset)
    pad_left, pad_right, pad_top = 120, 35, 35
    plot_width = width - pad_left - pad_right

    def sx(value):
        return pad_left + value * plot_width

    pieces = [
        f"<svg viewBox='0 0 {width} {height}' width='100%' height='{height}' role='img' aria-label='candidate supervised PR AUC ranges'>",
        "<rect width='100%' height='100%' fill='#ffffff'/>",
        f"<text x='{pad_left}' y='20' font-size='13' fill='#24292f'>Supervised PR AUC range across all seeds and candidates</text>",
    ]
    for tick in np.linspace(0, 1, 6):
        x = sx(tick)
        pieces.append(f"<line x1='{x:.1f}' y1='{pad_top}' x2='{x:.1f}' y2='{height-28}' stroke='#eaeef2'/>")
        pieces.append(f"<text x='{x:.1f}' y='{height-8}' text-anchor='middle' font-size='10' fill='#57606a'>{tick:.1f}</text>")
    for i, (dataset, row) in enumerate(by_dataset.iterrows()):
        y = pad_top + 30 + 48 * i
        pieces.append(f"<text x='12' y='{y+4}' font-size='12' fill='#24292f'>{escape(dataset)}</text>")
        pieces.append(f"<line x1='{sx(row.min_pr):.1f}' y1='{y}' x2='{sx(row.max_pr):.1f}' y2='{y}' stroke='#2f6f9f' stroke-width='7' stroke-linecap='round' opacity='.35'/>")
        pieces.append(f"<circle cx='{sx(row.mean_pr):.1f}' cy='{y}' r='6' fill='#2f6f9f'/>")
        pieces.append(f"<text x='{sx(row.max_pr)+6:.1f}' y='{y+4}' font-size='11' fill='#57606a'>max {row.max_pr:.2f}</text>")
    pieces.append("</svg>")
    return HtmlBlock("".join(pieces))

candidate_quality_svg(candidate_results)


"<svg viewBox='0 0 780 278' width='100%' height='278' role='img' aria-label='candidate supervised PR AUC ranges'><rect width='100%' height='100%' fill='#ffffff'/><text x='120' y='20' font-size='13' fill='#24292f'>Supervised PR AUC range across all seeds and candidates</text><line x1='120.0' y1='35' x2='120.0' y2='250' stroke='#eaeef2'/><text x='120.0' y='270' text-anchor='middle' font-size='10' fill='#57606a'>0.0</text><line x1='245.0' y1='35' x2='245.0' y2='250' stroke='#eaeef2'/><text x='245.0' y='270' text-anchor='middle' font-size='10' fill='#57606a'>0.2</text><line x1='370.0' y1='35' x2='370.0' y2='250' stroke='#eaeef2'/><text x='370.0' y='270' text-anchor='middle' font-size='10' fill='#57606a'>0.4</text><line x1='495.0' y1='35' x2='495.0' y2='250' stroke='#eaeef2'/><text x='495.0' y='270' text-anchor='middle' font-size='10' fill='#57606a'>0.6</text><line x1='620.0' y1='35' x2='620.0' y2='250' stroke='#eaeef2'/><text x='620.0' y='270' text-anchor='middle' font-size='10' fill='#57606a'>0.8</text><line x1='745.0' y1='35' x2='745.0' y2='250' stroke='#eaeef2'/><text x='745.0' y='270' text-anchor='middle' font-size='10' fill='#57606a'>1.0</text><text x='12' y='69' font-size='12' fill='#24292f'>cardio</text><line x1='661.4' y1='65' x2='739.4' y2='65' stroke='#2f6f9f' stroke-width='7' stroke-linecap='round' opacity='.35'/><circle cx='709.0' cy='65' r='6' fill='#2f6f9f'/><text x='745.4' y='69' font-size='11' fill='#57606a'>max 0.99</text><text x='12' y='117' font-size='12' fill='#24292f'>breastw</text><line x1='481.6' y1='113' x2='744.9' y2='113' stroke='#2f6f9f' stroke-width='7' stroke-linecap='round' opacity='.35'/><circle cx='711.6' cy='113' r='6' fill='#2f6f9f'/><text x='750.9' y='117' font-size='11' fill='#57606a'>max 1.00</text><text x='12' y='165' font-size='12' fill='#24292f'>satimage2</text><line x1='723.6' y1='161' x2='744.6' y2='161' stroke='#2f6f9f' stroke-width='7' stroke-linecap='round' opacity='.35'/><circle cx='740.4' cy='161' r='6' fill='#2f6f9f'/><text x='750.6' y='165' font-size='11' fill='#57606a'>max 1.00</text><text x='12' y='213' font-size='12' fill='#24292f'>annthyroid</text><line x1='638.3' y1='209' x2='737.6' y2='209' stroke='#2f6f9f' stroke-width='7' stroke-linecap='round' opacity='.35'/><circle cx='706.8' cy='209' r='6' fill='#2f6f9f'/><text x='743.6' y='213' font-size='11' fill='#57606a'>max 0.99</text></svg>"

## Signal metrics

A method has useful signal if it ranks candidates similarly to holdout quality, enriches the top-ranked subset, and avoids high-regret selections across splits.

In [7]:
def corr_or_nan(method, x, y):
    result = method(x, y)
    corr = result.statistic if hasattr(result, "statistic") else result[0]
    return float(corr) if np.isfinite(corr) else np.nan


def pairwise_agreement(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    agree = total = 0
    for i in range(len(x)):
        for j in range(i + 1, len(x)):
            dx = np.sign(x[i] - x[j])
            dy = np.sign(y[i] - y[j])
            if dx == 0 or dy == 0:
                continue
            total += 1
            agree += dx == dy
    return agree / total if total else np.nan


def within_model_spearman(group, signal_column):
    values = []
    for _, model_group in group.groupby("model"):
        if len(model_group) < 3 or model_group["pr_auc"].nunique() < 2:
            continue
        corr = corr_or_nan(spearmanr, model_group[signal_column], model_group["pr_auc"])
        if np.isfinite(corr):
            values.append(corr)
    return float(np.mean(values)) if values else np.nan


def between_model_spearman(group, signal_column):
    model_means = group.groupby("model")[[signal_column, "pr_auc"]].mean()
    if len(model_means) < 3 or model_means["pr_auc"].nunique() < 2:
        return np.nan
    return corr_or_nan(spearmanr, model_means[signal_column], model_means["pr_auc"])


def summarize_group(dataset, split_seed, group):
    rows = []
    best_pr = group["pr_auc"].max()
    pr_range = group["pr_auc"].max() - group["pr_auc"].min()
    mean_pr = group["pr_auc"].mean()
    top_n = max(3, math.ceil(len(group) * TOP_FRACTION))
    for method in metric_names:
        signal_column = f"signal_{method}"
        signal = group[signal_column]
        selected = group.loc[signal.idxmax()]
        top = group.loc[signal.nlargest(top_n).index]
        regret = best_pr - selected["pr_auc"]
        rows.append(
            {
                "dataset": dataset,
                "split_seed": split_seed,
                "method": method,
                "spearman_roc_auc": corr_or_nan(spearmanr, signal, group["roc_auc"]),
                "spearman_pr_auc": corr_or_nan(spearmanr, signal, group["pr_auc"]),
                "spearman_f1_at_k": corr_or_nan(spearmanr, signal, group["f1_at_k"]),
                "kendall_pr_auc": corr_or_nan(kendalltau, signal, group["pr_auc"]),
                "top20_pr_lift": top["pr_auc"].mean() - mean_pr,
                "selected_pr_auc": selected["pr_auc"],
                "regret_pr_auc": regret,
                "normalized_regret_pr_auc": 0.0 if pr_range == 0 else regret / pr_range,
                "pairwise_pr_agreement": pairwise_agreement(signal, group["pr_auc"]),
                "within_model_spearman_pr_auc": within_model_spearman(group, signal_column),
                "between_model_spearman_pr_auc": between_model_spearman(group, signal_column),
                "selected_model": selected["model"],
            }
        )
    return pd.DataFrame(rows)

signal_by_split = pd.concat(
    [
        summarize_group(dataset, split_seed, group)
        for (dataset, split_seed), group in candidate_results.groupby(["dataset", "split_seed"])
    ],
    ignore_index=True,
)
signal_by_split_shape = signal_by_split.shape


## Main evidence dashboard

Read this first. The cards combine the key checks: PR-AUC rank signal, bootstrap interval, top-20% enrichment, selection regret, pairwise agreement, and whether the method mostly separates detector families or tunes within a family.

In [8]:
summary_stats = [
    "spearman_pr_auc",
    "spearman_roc_auc",
    "spearman_f1_at_k",
    "top20_pr_lift",
    "regret_pr_auc",
    "pairwise_pr_agreement",
    "within_model_spearman_pr_auc",
    "between_model_spearman_pr_auc",
]


def mean_ci(values, rounds=BOOTSTRAP_ROUNDS, seed=BASE_SEED):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = np.empty(rounds, dtype=float)
    for i in range(rounds):
        means[i] = np.mean(rng.choice(values, size=values.size, replace=True))
    return float(values.mean()), float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))

summary_rows = []
for method, group in signal_by_split.groupby("method"):
    row = {"method": method}
    for stat in summary_stats:
        stable_seed = BASE_SEED + sum(ord(char) for char in f"{method}:{stat}")
        mean, low, high = mean_ci(group[stat], seed=stable_seed)
        row[stat] = mean
        row[f"{stat}_ci_low"] = low
        row[f"{stat}_ci_high"] = high
    summary_rows.append(row)

signal_summary = pd.DataFrame(summary_rows).sort_values(
    ["spearman_pr_auc", "regret_pr_auc"], ascending=[False, True]
).reset_index(drop=True)
summary_view = signal_summary[
    [
        "method",
        "spearman_pr_auc",
        "spearman_pr_auc_ci_low",
        "spearman_pr_auc_ci_high",
        "top20_pr_lift",
        "regret_pr_auc",
        "pairwise_pr_agreement",
        "within_model_spearman_pr_auc",
        "between_model_spearman_pr_auc",
    ]
]

class HtmlTable(str):
    def _repr_html_(self):
        return str(self)


def pct(value):
    return "n/a" if pd.isna(value) else f"{value:.2f}"


main_dashboard = signal_summary.head(8).assign(
    pr_signal_ci=lambda frame: frame.apply(
        lambda row: f"{row.spearman_pr_auc:.2f} [{row.spearman_pr_auc_ci_low:.2f}, {row.spearman_pr_auc_ci_high:.2f}]",
        axis=1,
    ),
    top20_pr_lift=lambda frame: frame["top20_pr_lift"].round(3),
    regret_pr_auc=lambda frame: frame["regret_pr_auc"].round(3),
    pairwise_pr_agreement=lambda frame: frame["pairwise_pr_agreement"].round(2),
)[["method", "pr_signal_ci", "top20_pr_lift", "regret_pr_auc", "pairwise_pr_agreement"]]
main_dashboard


,method,pr_signal_ci,top20_pr_lift,regret_pr_auc,pairwise_pr_agreement
0,excess_mass,"0.53 [0.40, 0.65]",0.017,0.010,0.72
1,mass_volume,"0.49 [0.33, 0.64]",0.015,0.011,0.71
2,npd,"0.35 [0.19, 0.50]",0.013,0.011,0.65
3,sireos,"0.27 [0.15, 0.38]",0.009,0.016,0.60
4,eag,"0.14 [0.02, 0.25]",-0.010,0.033,0.56
5,ireos,"0.09 [-0.06, 0.25]",0.012,0.016,0.54
6,score_silhouette,"0.05 [-0.11, 0.21]",-0.023,0.074,0.52
7,rtm,"0.04 [-0.03, 0.10]",-0.005,0.090,0.52


## Dataset consistency

The heatmap shows where signal is stable. A good selector should not only win on one dataset.

In [9]:
per_dataset_signal = signal_by_split.groupby(["dataset", "method"], as_index=False).agg(
    spearman_pr_auc=("spearman_pr_auc", "mean"),
    regret_pr_auc=("regret_pr_auc", "mean"),
    top20_pr_lift=("top20_pr_lift", "mean"),
)


def color_cell(value, scale=1.0, positive="68, 119, 170", negative="204, 102, 119", digits=2):
    if pd.isna(value):
        return ""
    alpha = min(abs(float(value)) / scale, 1.0) * 0.72
    color = positive if value >= 0 else negative
    return f'<span style="display:block; padding:4px 8px; background-color: rgba({color}, {alpha:.2f}); color: #000">{value:.{digits}f}</span>'


def html_table(df, index=True):
    return HtmlBlock(df.to_html(escape=False, index=index))

heat = per_dataset_signal.pivot(index="method", columns="dataset", values="spearman_pr_auc").loc[
    signal_summary["method"].head(12)
]
html_table(heat.map(lambda value: color_cell(value, scale=0.8)))


dataset,annthyroid,breastw,cardio,satimage2
method,,,,
excess_mass,0.54,0.90,-0.08,0.75
mass_volume,0.47,0.92,-0.18,0.75
npd,-0.10,0.91,-0.13,0.73
sireos,0.70,-0.04,0.02,0.41
eag,0.22,0.02,0.19,0.13
ireos,0.34,0.08,0.30,-0.37
score_silhouette,0.57,-0.70,0.17,0.16
rtm,0.13,0.02,-0.19,0.20
asoi,-0.07,0.64,0.04,-0.57


## Regret by dataset

Regret is the tuner-facing view: if this method were used to choose a candidate, how far would it land from the best labeled PR AUC in that split? Lower is better.

In [10]:
regret_heat = per_dataset_signal.pivot(index="method", columns="dataset", values="regret_pr_auc").loc[
    signal_summary["method"].head(12)
]
html_table(regret_heat.map(lambda value: color_cell(-value, scale=0.12, positive="68, 119, 170", negative="204, 102, 119", digits=3)))


dataset,annthyroid,breastw,cardio,satimage2
method,,,,
excess_mass,-0.011,-0.001,-0.025,-0.002
mass_volume,-0.015,-0.001,-0.025,-0.003
npd,-0.016,-0.004,-0.022,-0.001
sireos,-0.005,-0.043,-0.014,-0.003
eag,-0.018,-0.100,-0.011,-0.002
ireos,-0.012,-0.017,-0.024,-0.010
score_silhouette,-0.011,-0.261,-0.021,-0.004
rtm,-0.012,-0.317,-0.029,-0.000
asoi,-0.041,-0.035,-0.026,-0.014


## Signal versus regret

The best region is upper-left: high PR-AUC rank signal and low selection regret. Ensembles are included because the single-metric objective can be brittle.

In [11]:
def scatter_svg(summary):
    frame = summary.copy()
    width, height = 780, 420
    pad_left, pad_right, pad_top, pad_bottom = 80, 30, 35, 60
    x_min, x_max = -0.15, max(0.65, frame["spearman_pr_auc"].max() + 0.06)
    y_min, y_max = 0.0, max(0.14, frame["regret_pr_auc"].max() + 0.02)

    def sx(x):
        return pad_left + (x - x_min) / (x_max - x_min) * (width - pad_left - pad_right)

    def sy(y):
        return height - pad_bottom - (y - y_min) / (y_max - y_min) * (height - pad_top - pad_bottom)

    pieces = [
        f"<svg viewBox='0 0 {width} {height}' width='100%' height='{height}' role='img' aria-label='signal versus regret scatter plot'>",
        "<rect width='100%' height='100%' fill='#ffffff'/>",
        f"<line x1='{pad_left}' y1='{height-pad_bottom}' x2='{width-pad_right}' y2='{height-pad_bottom}' stroke='#8c959f'/>",
        f"<line x1='{pad_left}' y1='{pad_top}' x2='{pad_left}' y2='{height-pad_bottom}' stroke='#8c959f'/>",
        f"<text x='{width/2}' y='{height-16}' text-anchor='middle' font-size='13' fill='#24292f'>Spearman(PR AUC), higher is better</text>",
        f"<text x='18' y='{height/2}' transform='rotate(-90 18 {height/2})' text-anchor='middle' font-size='13' fill='#24292f'>Selection regret, lower is better</text>",
        f"<rect x='{sx(0.30)}' y='{pad_top}' width='{width-pad_right-sx(0.30)}' height='{sy(0.03)-pad_top}' fill='rgba(63,127,95,.10)'/>",
        f"<text x='{sx(0.31)}' y='{pad_top+16}' font-size='12' fill='#3f7f5f'>promising region</text>",
    ]
    for _, row in frame.iterrows():
        x = sx(row["spearman_pr_auc"])
        y = sy(row["regret_pr_auc"])
        pieces.append(f"<circle cx='{x:.1f}' cy='{y:.1f}' r='6' fill='#2f6f9f' opacity='0.85'/>")
    for _, row in frame.head(8).iterrows():
        x = sx(row["spearman_pr_auc"]) + 9
        y = sy(row["regret_pr_auc"]) - 7
        pieces.append(f"<text x='{x:.1f}' y='{y:.1f}' font-size='11' fill='#24292f'>{escape(row['method'])}</text>")
    pieces.append("</svg>")
    return HtmlBlock("".join(pieces))

scatter_svg(signal_summary)


"<svg viewBox='0 0 780 420' width='100%' height='420' role='img' aria-label='signal versus regret scatter plot'><rect width='100%' height='100%' fill='#ffffff'/><line x1='80' y1='360' x2='750' y2='360' stroke='#8c959f'/><line x1='80' y1='35' x2='80' y2='360' stroke='#8c959f'/><text x='390.0' y='404' text-anchor='middle' font-size='13' fill='#24292f'>Spearman(PR AUC), higher is better</text><text x='18' y='210.0' transform='rotate(-90 18 210.0)' text-anchor='middle' font-size='13' fill='#24292f'>Selection regret, lower is better</text><rect x='456.87499999999994' y='35' width='293.12500000000006' height='255.3571428571429' fill='rgba(63,127,95,.10)'/><text x='465.24999999999994' y='51' font-size='12' fill='#3f7f5f'>promising region</text><circle cx='647.6' cy='336.9' r='6' fill='#2f6f9f' opacity='0.85'/><circle cx='615.6' cy='334.9' r='6' fill='#2f6f9f' opacity='0.85'/><circle cx='500.2' cy='334.6' r='6' fill='#2f6f9f' opacity='0.85'/><circle cx='435.5' cy='322.4' r='6' fill='#2f6f9f' opacity='0.85'/><circle cx='321.8' cy='283.7' r='6' fill='#2f6f9f' opacity='0.85'/><circle cx='277.8' cy='323.5' r='6' fill='#2f6f9f' opacity='0.85'/><circle cx='248.1' cy='187.6' r='6' fill='#2f6f9f' opacity='0.85'/><circle cx='238.9' cy='151.9' r='6' fill='#2f6f9f' opacity='0.85'/><circle cx='213.8' cy='292.9' r='6' fill='#2f6f9f' opacity='0.85'/><circle cx='168.6' cy='308.5' r='6' fill='#2f6f9f' opacity='0.85'/><text x='656.6' y='329.9' font-size='11' fill='#24292f'>excess_mass</text><text x='624.6' y='327.9' font-size='11' fill='#24292f'>mass_volume</text><text x='509.2' y='327.6' font-size='11' fill='#24292f'>npd</text><text x='444.5' y='315.4' font-size='11' fill='#24292f'>sireos</text><text x='330.8' y='276.7' font-size='11' fill='#24292f'>eag</text><text x='286.8' y='316.5' font-size='11' fill='#24292f'>ireos</text><text x='257.1' y='180.6' font-size='11' fill='#24292f'>score_silhouette</text><text x='247.9' y='144.9' font-size='11' fill='#24292f'>rtm</text></svg>"

## What to take away

This deliberately compresses the numerical audit into an actionable read: strong enough to consider, low-regret but noisy, mixed, or weak.

In [12]:
def recommendation(row):
    if row["spearman_pr_auc"] >= 0.30 and row["regret_pr_auc"] <= 0.03 and row["pairwise_pr_agreement"] >= 0.58:
        return "promising weak selector"
    if row["regret_pr_auc"] <= 0.03 and row["spearman_pr_auc"] > 0.10:
        return "low-regret but noisy"
    if row["spearman_pr_auc"] <= 0.05:
        return "weak or unstable"
    return "mixed signal"

interpretation = signal_summary.copy()
interpretation["assessment"] = interpretation.apply(recommendation, axis=1)


def takeaways_html(frame):
    rows = []
    for row in frame.head(12).itertuples(index=False):
        rows.append(
            f"""
            <tr style='border-bottom:1px solid #eaeef2;'>
              <td><span style='display:inline-block; width:10px; height:10px; border-radius:50%; background:#2f6f9f; margin-right:6px;'></span><b>{escape(row.method)}</b></td>
              <td>{escape(row.assessment)}</td>
              <td>{row.spearman_pr_auc:.2f}</td>
              <td>{row.regret_pr_auc:.3f}</td>
              <td>{row.pairwise_pr_agreement:.2f}</td>
            </tr>
            """
        )
    return HtmlBlock(
        """
        <table style='border-collapse:collapse; width:100%; font-size:13px; color:#000; background:#fff;'>
          <thead><tr style='border-bottom:1px solid #d8dee4; text-align:left;'><th>Method</th><th>Assessment</th><th>Signal</th><th>Regret</th><th>Agreement</th></tr></thead>
          <tbody>
        """ + "\n".join(rows) + "</tbody></table>"
    )

takeaways_html(interpretation)


Method,Assessment,Signal,Regret,Agreement
excess_mass,promising weak selector,0.53,0.010,0.72
mass_volume,promising weak selector,0.49,0.011,0.71
npd,promising weak selector,0.35,0.011,0.65
sireos,low-regret but noisy,0.27,0.016,0.60
eag,mixed signal,0.14,0.033,0.56
ireos,mixed signal,0.09,0.016,0.54
score_silhouette,mixed signal,0.05,0.074,0.52
rtm,weak or unstable,0.04,0.090,0.52
asoi,weak or unstable,0.01,0.029,0.51
laplacian,weak or unstable,-0.04,0.022,0.49


## Appendix: compact numeric audit

The detailed numeric view is kept short and native. It is aggregated over all seeds; no single-seed rows are shown.

In [13]:
summary_view.head(12)[
    ["method", "spearman_pr_auc", "top20_pr_lift", "regret_pr_auc", "pairwise_pr_agreement"]
].round(3)


,method,spearman_pr_auc,top20_pr_lift,regret_pr_auc,pairwise_pr_agreement
0,excess_mass,0.528,0.017,0.010,0.718
1,mass_volume,0.489,0.015,0.011,0.708
2,npd,0.352,0.013,0.011,0.650
3,sireos,0.274,0.009,0.016,0.605
4,eag,0.139,-0.010,0.033,0.556
5,ireos,0.086,0.012,0.016,0.537
6,score_silhouette,0.051,-0.023,0.074,0.519
7,rtm,0.040,-0.005,0.090,0.516
8,asoi,0.010,0.006,0.029,0.505
9,laplacian,-0.044,0.006,0.022,0.491


## Model-family diagnostics

If between-model signal is much stronger than within-model signal, the method mostly picks detector families rather than tuning hyperparameters inside a family.

In [14]:
family_view = signal_summary[
    ["method", "within_model_spearman_pr_auc", "between_model_spearman_pr_auc", "spearman_pr_auc"]
].copy()
family_view["family_minus_within"] = family_view["between_model_spearman_pr_auc"] - family_view["within_model_spearman_pr_auc"]
family_view = family_view.sort_values("spearman_pr_auc", ascending=False).head(12)


def family_dumbbell_svg(frame):
    width, height = 900, 70 + 36 * len(frame)
    pad_left, pad_right, pad_top, pad_bottom = 260, 45, 34, 36
    x_min, x_max = -0.25, 0.70

    def sx(value):
        return pad_left + (value - x_min) / (x_max - x_min) * (width - pad_left - pad_right)

    pieces = [
        f"<svg viewBox='0 0 {width} {height}' width='100%' height='{height}' role='img' aria-label='within versus between model signal'>",
        "<rect width='100%' height='100%' fill='#ffffff'/>",
        f"<text x='{pad_left}' y='20' font-size='13' fill='#24292f'>Within-model versus between-model PR-AUC signal</text>",
        f"<text x='16' y='20' font-size='11' fill='#57606a'>gray = within family, color = between families</text>",
    ]
    for tick in [-0.2, 0.0, 0.2, 0.4, 0.6]:
        x = sx(tick)
        pieces.append(f"<line x1='{x:.1f}' y1='{pad_top}' x2='{x:.1f}' y2='{height-pad_bottom}' stroke='#eaeef2'/>")
        pieces.append(f"<text x='{x:.1f}' y='{height-10}' text-anchor='middle' font-size='10' fill='#57606a'>{tick:.1f}</text>")
    for i, row in enumerate(frame.itertuples(index=False)):
        y = pad_top + 28 + i * 34
        within = row.within_model_spearman_pr_auc
        between = row.between_model_spearman_pr_auc
        label = escape(row.method)
        pieces.append(f"<text x='16' y='{y+4}' font-size='11' fill='#24292f'>{label}</text>")
        pieces.append(f"<line x1='{sx(within):.1f}' y1='{y}' x2='{sx(between):.1f}' y2='{y}' stroke='#8c959f' stroke-width='2'/>")
        pieces.append(f"<circle cx='{sx(within):.1f}' cy='{y}' r='5' fill='#d0d7de'/>")
        pieces.append(f"<circle cx='{sx(between):.1f}' cy='{y}' r='6' fill='#2f6f9f'/>")
    pieces.append("</svg>")
    return HtmlBlock("".join(pieces))

family_dumbbell_svg(family_view)


"<svg viewBox='0 0 900 430' width='100%' height='430' role='img' aria-label='within versus between model signal'><rect width='100%' height='100%' fill='#ffffff'/><text x='260' y='20' font-size='13' fill='#24292f'>Within-model versus between-model PR-AUC signal</text><text x='16' y='20' font-size='11' fill='#57606a'>gray = within family, color = between families</text><line x1='291.3' y1='34' x2='291.3' y2='394' stroke='#eaeef2'/><text x='291.3' y='420' text-anchor='middle' font-size='10' fill='#57606a'>-0.2</text><line x1='416.6' y1='34' x2='416.6' y2='394' stroke='#eaeef2'/><text x='416.6' y='420' text-anchor='middle' font-size='10' fill='#57606a'>0.0</text><line x1='541.8' y1='34' x2='541.8' y2='394' stroke='#eaeef2'/><text x='541.8' y='420' text-anchor='middle' font-size='10' fill='#57606a'>0.2</text><line x1='667.1' y1='34' x2='667.1' y2='394' stroke='#eaeef2'/><text x='667.1' y='420' text-anchor='middle' font-size='10' fill='#57606a'>0.4</text><line x1='792.4' y1='34' x2='792.4' y2='394' stroke='#eaeef2'/><text x='792.4' y='420' text-anchor='middle' font-size='10' fill='#57606a'>0.6</text><text x='16' y='66' font-size='11' fill='#24292f'>excess_mass</text><line x1='591.7' y1='62' x2='782.1' y2='62' stroke='#8c959f' stroke-width='2'/><circle cx='591.7' cy='62' r='5' fill='#d0d7de'/><circle cx='782.1' cy='62' r='6' fill='#2f6f9f'/><text x='16' y='100' font-size='11' fill='#24292f'>mass_volume</text><line x1='543.9' y1='96' x2='718.9' y2='96' stroke='#8c959f' stroke-width='2'/><circle cx='543.9' cy='96' r='5' fill='#d0d7de'/><circle cx='718.9' cy='96' r='6' fill='#2f6f9f'/><text x='16' y='134' font-size='11' fill='#24292f'>npd</text><line x1='617.2' y1='130' x2='572.7' y2='130' stroke='#8c959f' stroke-width='2'/><circle cx='617.2' cy='130' r='5' fill='#d0d7de'/><circle cx='572.7' cy='130' r='6' fill='#2f6f9f'/><text x='16' y='168' font-size='11' fill='#24292f'>sireos</text><line x1='652.4' y1='164' x2='589.4' y2='164' stroke='#8c959f' stroke-width='2'/><circle cx='652.4' cy='164' r='5' fill='#d0d7de'/><circle cx='589.4' cy='164' r='6' fill='#2f6f9f'/><text x='16' y='202' font-size='11' fill='#24292f'>eag</text><line x1='574.1' y1='198' x2='409.3' y2='198' stroke='#8c959f' stroke-width='2'/><circle cx='574.1' cy='198' r='5' fill='#d0d7de'/><circle cx='409.3' cy='198' r='6' fill='#2f6f9f'/><text x='16' y='236' font-size='11' fill='#24292f'>ireos</text><line x1='585.3' y1='232' x2='464.4' y2='232' stroke='#8c959f' stroke-width='2'/><circle cx='585.3' cy='232' r='5' fill='#d0d7de'/><circle cx='464.4' cy='232' r='6' fill='#2f6f9f'/><text x='16' y='270' font-size='11' fill='#24292f'>score_silhouette</text><line x1='535.0' y1='266' x2='417.6' y2='266' stroke='#8c959f' stroke-width='2'/><circle cx='535.0' cy='266' r='5' fill='#d0d7de'/><circle cx='417.6' cy='266' r='6' fill='#2f6f9f'/><text x='16' y='304' font-size='11' fill='#24292f'>rtm</text><line x1='384.9' y1='300' x2='476.9' y2='300' stroke='#8c959f' stroke-width='2'/><circle cx='384.9' cy='300' r='5' fill='#d0d7de'/><circle cx='476.9' cy='300' r='6' fill='#2f6f9f'/><text x='16' y='338' font-size='11' fill='#24292f'>asoi</text><line x1='516.9' y1='334' x2='375.3' y2='334' stroke='#8c959f' stroke-width='2'/><circle cx='516.9' cy='334' r='5' fill='#d0d7de'/><circle cx='375.3' cy='334' r='6' fill='#2f6f9f'/><text x='16' y='372' font-size='11' fill='#24292f'>laplacian</text><line x1='448.2' y1='368' x2='383.8' y2='368' stroke='#8c959f' stroke-width='2'/><circle cx='448.2' cy='368' r='5' fill='#d0d7de'/><circle cx='383.8' cy='368' r='6' fill='#2f6f9f'/></svg>"